In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [43:44<00:00, 65.0kB/s]


In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [7]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3814646927139642
epoch=2/10 & loss=0.9488581761984569
epoch=3/10 & loss=0.7545047124938282
epoch=4/10 & loss=0.6309561085746721
epoch=5/10 & loss=0.5204820364256344
epoch=6/10 & loss=0.42885651992028934
epoch=7/10 & loss=0.3523345240546614
epoch=8/10 & loss=0.2735684307201592
epoch=9/10 & loss=0.21593649943580712
epoch=10/10 & loss=0.1698930824504179


In [8]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.76


### Detailed Evaluation Metrics (Precision, Recall, F1, Confusion Matrix)

In [9]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

classes = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=classes))

              precision    recall  f1-score   support

       plane       0.78      0.82      0.80      1000
         car       0.84      0.89      0.86      1000
        bird       0.63      0.68      0.65      1000
         cat       0.60      0.46      0.52      1000
        deer       0.73      0.68      0.71      1000
         dog       0.59      0.76      0.66      1000
        frog       0.84      0.76      0.80      1000
       horse       0.77      0.80      0.78      1000
        ship       0.86      0.85      0.85      1000
       truck       0.87      0.79      0.82      1000

    accuracy                           0.75     10000
   macro avg       0.75      0.75      0.75     10000
weighted avg       0.75      0.75      0.75     10000



In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - CIFAR10 CNN")
plt.tight_layout()
plt.savefig("confusion_matrix_cifar10.png", dpi=150)
plt.show()

### Per-Class Accuracy
Ee cell class-wise ela perform avutundo chupistundi — some classes (e.g. cat vs dog) confuse avvachu, avi easy ga ikkada kanapadthayi.

In [11]:
class_correct = {c: 0 for c in classes}
class_total = {c: 0 for c in classes}

for true_label, pred_label in zip(all_labels, all_preds):
    cname = classes[true_label]
    class_total[cname] += 1
    if true_label == pred_label:
        class_correct[cname] += 1

for c in classes:
    acc = 100 * class_correct[c] / class_total[c]
    print(f"{c:8s}: {acc:.2f}%")

plane   : 81.90%
car     : 89.00%
bird    : 67.60%
cat     : 45.50%
deer    : 68.40%
dog     : 75.80%
frog    : 76.10%
horse   : 79.80%
ship    : 84.80%
truck   : 78.70%
